In [2]:
import numpy as np
# version: numpy 2.3.4
from scipy.stats import norm
# version: scipy 1.16.2

In [18]:
'''
estimate conditional entropy
'''
# variance of measured noise
var_m = 10.6
# variance of electronic noise
var_e = 3.1
std_e = np.sqrt(var_e)
# max_input = 10 ** (2/20)            # 2dBV
# bin_width = max_input / 2 ** 15     # 16-bit sampling

bin = np.linspace(start=-2 ** 15, stop=2 ** 15 - 1, num=2 ** 16, dtype=int)
# lower bound of ADC bins
bin_low_bound = bin - 0.5
bin_low_bound[0] = -1 * np.inf

# higher bound of ADC bins
bin_high_bound = bin + 0.5
bin_high_bound[-1] = np.inf

# probability mass function of e noise
P_Edis = norm.cdf(bin_high_bound / std_e) - norm.cdf(bin_low_bound / std_e)

# probability mass function of measured noise conditioned on e noise, i.e., quantum noise
# here variance is var_q
var_q = var_m - var_e
std_q = np.sqrt(var_q)
P_Mdis_E = norm.cdf(bin_high_bound / std_q) - norm.cdf(bin_low_bound / std_q)

# We can easily notice that the max(P_Mdis|E) = P_Mdis|E(e|e) under wide bin width.
# process of optimizing the range R or the amplification times is ignored temporarily.
P_Mdis_Edis_max = P_Mdis_E[bin == 0]

# useful range of E
bound = int(10 * std_q) + 1

# P_guess
P_guess = np.sum(P_Edis[(bin > -1*bound) & (bin < bound)] * P_Mdis_Edis_max)
# print(Pguess_Mdis_Edis)
Hmin_Mdis_Edis = -np.log2(P_guess)
print(f'entropy: {Hmin_Mdis_Edis}')

entropy: 2.787190542818872


In [19]:
'''
estimate bits of output of each extraction
'''
CHUNK = 256
ep = 2 ** -50
print(int(CHUNK * Hmin_Mdis_Edis) // 1)
extractable_bits = int(CHUNK * Hmin_Mdis_Edis // 1 + 2 * np.log2(ep))
print(extractable_bits)

713
613


In [13]:
'''
estimate extraction time
'''
bit_in_10s = int(48000 / CHUNK * extractable_bits * 10)
print(bit_in_10s)
time = 1e9 / bit_in_10s * 10
print(time)

1149375
8700.380641653072
